# Distilled Image Encoder Interactive Eval

Run ES-RV-S/M/L timing and overlay checks with text, point, or box prompts. Outputs are written under `RUN_ROOT/eval/notebook_interactive`, not under the repo root.

In [ ]:
from pathlib import Path
import os, sys, time

REPO_DIR = Path(os.environ.get('REPO_DIR', '/storage/project/r-agarg35-0/eliu354/projects/EfficientSam3-Distillation')).resolve()
RUN_ROOT = Path(os.environ.get('RUN_ROOT', '/storage/scratch1/9/eliu354/efficientsam3_distill_smoke')).resolve()
COCO_ROOT = RUN_ROOT / 'data' / 'coco_fixed10'
OUT_DIR = RUN_ROOT / 'eval' / 'notebook_interactive'
BENCHMARK_ROOT = Path('/storage/home/hcoda1/9/eliu354/r-agarg35-0/projects/efficientsam3-benchmark')

sys.path.insert(0, str(REPO_DIR))
from tools.eval_distilled_image_encoder import (
    MODEL_SPECS, FIXED10_MANIFEST, FIXED10_PROMPTS,
    copy_or_download_fixed10, load_fixed10_examples, load_model,
    predict_prompt, calculate_iou, save_overlay,
)

print('REPO_DIR =', REPO_DIR)
print('RUN_ROOT =', RUN_ROOT)
print('OUT_DIR =', OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Prepare exactly the fixed 10 COCO images used by the benchmark repo.
copy_or_download_fixed10(COCO_ROOT, BENCHMARK_ROOT, FIXED10_MANIFEST, FIXED10_PROMPTS)
examples = load_fixed10_examples(COCO_ROOT, FIXED10_MANIFEST)
print('COCO fixed10 images:', len(examples))
for i, ex in enumerate(examples):
    print(i, ex.sample_id, ex.category, ex.text_prompt, ex.image_path.name)

In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw
from IPython.display import display, Markdown, Image as IPyImage
import ipywidgets as widgets

from sam3.device import get_device
from sam3.model.sam3_image_processor import Sam3Processor

device = get_device()
print('device:', device)
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

model_cache = {}

def checkpoint_for(size):
    return RUN_ROOT / 'output' / MODEL_SPECS[size]['checkpoint']

def get_model(size):
    if size not in model_cache:
        ckpt = checkpoint_for(size)
        print('loading', MODEL_SPECS[size]['label'], ckpt)
        t0 = time.perf_counter()
        model_cache[size] = load_model(size, ckpt, device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        print(f'loaded in {time.perf_counter() - t0:.3f}s')
    return model_cache[size]

def draw_prompt_preview(image, point, box, text):
    canvas = image.convert('RGBA')
    draw = ImageDraw.Draw(canvas)
    x1, y1, x2, y2 = box
    draw.rectangle([x1, y1, x2, y2], outline=(40, 255, 40, 255), width=4)
    x, y = point
    r = 7
    draw.ellipse([x-r, y-r, x+r, y+r], fill=(255, 220, 0, 255), outline=(0, 0, 0, 255), width=2)
    draw.line([x-14, y, x+14, y], fill=(255, 220, 0, 255), width=2)
    draw.line([x, y-14, x, y+14], fill=(255, 220, 0, 255), width=2)
    draw.rectangle([0, 0, min(canvas.width, 900), 30], fill=(0, 0, 0, 170))
    draw.text((8, 8), f'text={text} | point=({x:.1f},{y:.1f}) | box=({x1:.1f},{y1:.1f},{x2:.1f},{y2:.1f})', fill=(255,255,255,255))
    return canvas.convert('RGB')

In [ ]:
size_w = widgets.Dropdown(options=[('ES-RV-S','s'), ('ES-RV-M','m'), ('ES-RV-L','l')], value='m', description='model')
prompt_w = widgets.Dropdown(options=['text', 'point', 'box'], value='box', description='prompt')
sample_w = widgets.IntSlider(min=0, max=len(examples)-1, value=0, step=1, description='sample')
text_w = widgets.Text(value=examples[0].text_prompt, description='text')
px_w = widgets.FloatSlider(min=0, max=640, value=float(examples[0].point[0]), step=1, description='point x')
py_w = widgets.FloatSlider(min=0, max=640, value=float(examples[0].point[1]), step=1, description='point y')
x1_w = widgets.FloatSlider(min=0, max=640, value=float(examples[0].box_xyxy[0]), step=1, description='box x1')
y1_w = widgets.FloatSlider(min=0, max=640, value=float(examples[0].box_xyxy[1]), step=1, description='box y1')
x2_w = widgets.FloatSlider(min=0, max=640, value=float(examples[0].box_xyxy[2]), step=1, description='box x2')
y2_w = widgets.FloatSlider(min=0, max=640, value=float(examples[0].box_xyxy[3]), step=1, description='box y2')
preview_out = widgets.Output()
result_out = widgets.Output()
run_btn = widgets.Button(description='Run prompt', button_style='primary')

def load_sample(index):
    ex = examples[index]
    image = Image.open(ex.image_path).convert('RGB')
    px_w.max = x1_w.max = x2_w.max = image.width
    py_w.max = y1_w.max = y2_w.max = image.height
    text_w.value = ex.text_prompt
    px_w.value, py_w.value = map(float, ex.point)
    x1_w.value, y1_w.value, x2_w.value, y2_w.value = map(float, ex.box_xyxy)
    update_preview()

def update_preview(*_):
    ex = examples[sample_w.value]
    image = Image.open(ex.image_path).convert('RGB')
    point = np.array([px_w.value, py_w.value], dtype=np.float32)
    box = np.array([x1_w.value, y1_w.value, x2_w.value, y2_w.value], dtype=np.float32)
    preview = draw_prompt_preview(image, point, box, text_w.value)
    with preview_out:
        preview_out.clear_output(wait=True)
        display(Markdown(f'**{ex.sample_id}** category=`{ex.category}`; red GT appears only in result overlay.'))
        display(preview.resize((min(900, preview.width), int(preview.height * min(900, preview.width) / preview.width))))

def on_sample_change(change):
    load_sample(change['new'])

sample_w.observe(on_sample_change, names='value')
for w in [text_w, px_w, py_w, x1_w, y1_w, x2_w, y2_w]:
    w.observe(update_preview, names='value')

def on_run(_):
    ex = examples[sample_w.value]
    image = Image.open(ex.image_path).convert('RGB')
    point = np.array([px_w.value, py_w.value], dtype=np.float32)
    box = np.array([x1_w.value, y1_w.value, x2_w.value, y2_w.value], dtype=np.float32)
    model = get_model(size_w.value)
    processor = Sam3Processor(model, device=device)
    mask, score, timing = predict_prompt(
        model, processor, image, prompt_w.value, device,
        threshold=0.0, multimask_output=False,
        box_xyxy=box, point=point, point_label=1, text_prompt=text_w.value,
    )
    iou = calculate_iou(mask, ex.gt_mask)
    overlay_path = OUT_DIR / f'{MODEL_SPECS[size_w.value]["label"].lower()}_{prompt_w.value}_{ex.sample_id}.jpg'
    save_overlay(
        image, mask, overlay_path,
        box_xyxy=box if prompt_w.value == 'box' else None,
        point=point if prompt_w.value == 'point' else None,
        gt_mask=ex.gt_mask,
        title=f'{MODEL_SPECS[size_w.value]["label"]} {prompt_w.value} IoU={iou:.3f} total={timing["total_sec"]:.3f}s',
    )
    with result_out:
        result_out.clear_output(wait=True)
        display(Markdown(f'**time:** total `{timing["total_sec"]:.3f}s`, set_image `{timing["set_image_sec"]:.3f}s`, prompt `{timing["prompt_sec"]:.3f}s`  \n**IoU:** `{iou:.4f}`  \n**score:** `{score:.4f}`  \n**overlay:** `{overlay_path}`'))
        display(IPyImage(filename=str(overlay_path)))

run_btn.on_click(on_run)
load_sample(0)
display(widgets.VBox([
    widgets.HBox([size_w, prompt_w, sample_w]),
    text_w,
    widgets.HBox([px_w, py_w]),
    widgets.HBox([x1_w, y1_w]),
    widgets.HBox([x2_w, y2_w]),
    preview_out,
    run_btn,
    result_out,
]))

In [ ]:
# Optional batch CLI run from the notebook: all three sizes x text/point/box on fixed COCO-10.
# This writes metrics.csv, summary.json, and overlays under RUN_ROOT/eval/image_encoder_distill/<run-name>/.
# !python {REPO_DIR}/tools/eval_distilled_image_encoder.py --prepare-coco10 --run-root {RUN_ROOT} --prompt-modes text point box